### 14. Perfiles con mayor probabilidad de ganar medalla

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from cargue_datos import cargar_datos, preparar_datos

df_atletas, df_regiones = cargar_datos()
df_atletas = preparar_datos(df_atletas, df_regiones)

df_atletas['grupo_etario'] = pd.cut(
    df_atletas['Age'],
    bins=[0, 17, 34, df_atletas['Age'].max()],
    labels=['Menores', 'Jóvenes adultos', 'Mayores'])
df_atletas['grupo_etario'] = df_atletas['grupo_etario'].cat.add_categories('Sin edad').fillna('Sin edad')

df_atletas['n_pruebas'] = df_atletas.groupby(['ID', 'Games'])['Event'].transform('count')
df_atletas['categoria_pruebas'] = pd.cut(
    df_atletas['n_pruebas'],
    bins=[0, 1, 3, df_atletas['n_pruebas'].max()],
    labels=['1 prueba', '2-3 pruebas', '4 o más pruebas'])

print(df_atletas.shape)

(271116, 22)


In [2]:
def tasa_medalla(columna):
    t = df_atletas.groupby(columna, observed=True)['medalla'].agg(medallas='sum', registros='count')
    t['tasa_pct'] = (t['medallas'] / t['registros'] * 100).round(2)
    return t

display(tasa_medalla('Sex'))
display(tasa_medalla('grupo_etario'))
display(tasa_medalla('Season'))
display(tasa_medalla('categoria_pruebas'))

paises = tasa_medalla('region')
paises = paises[paises['registros'] >= 1000].sort_values('tasa_pct', ascending=False)
display(paises.head(10))

perfil = tasa_medalla(['Sex', 'grupo_etario']).sort_values('tasa_pct', ascending=False)
display(perfil)

paises.head(10).to_csv('../outputs/tablas/probabilidad_medalla_por_pais.csv')
perfil.to_csv('../outputs/tablas/probabilidad_medalla_por_perfil.csv')

,medallas,registros,tasa_pct
Sex,,,
F,11253,74522,15.10
M,28530,196594,14.51


,medallas,registros,tasa_pct
grupo_etario,,,
Menores,1179,12508,9.43
Jóvenes adultos,35066,229617,15.27
Mayores,2806,19517,14.38
Sin edad,732,9474,7.73


,medallas,registros,tasa_pct
Season,,,
Summer,34088,222552,15.32
Winter,5695,48564,11.73


,medallas,registros,tasa_pct
categoria_pruebas,,,
1 prueba,24464,139986,17.48
2-3 pruebas,11281,88160,12.80
4 o más pruebas,4038,42970,9.40


,medallas,registros,tasa_pct
region,,,
Russia,3947,11692,33.76
USA,5637,18853,29.90
Germany,3756,15883,23.65
Norway,1033,4960,20.83
Sweden,1536,8339,18.42
Netherlands,1040,5839,17.81
Australia,1349,7724,17.47
Hungary,1135,6607,17.18
China,993,5826,17.04


medallas  registros  tasa_pct
Sex grupo_etario                                  
F   Jóvenes adultos      9951      61889     16.08
M   Jóvenes adultos     25115     167728     14.97
    Mayores              2408      16644     14.47
F   Mayores               398       2873     13.85
    Menores               887       9336      9.50
M   Menores               292       3172      9.21
    Sin edad              715       9050      7.90
F   Sin edad               17        424      4.01

### Conclusión: perfiles con mayor probabilidad de medalla

Se midió la probabilidad de medalla con la tasa (medallas / registros), para no depender del tamaño de cada grupo.

- **País:** es el factor con mayores diferencias. Entre los países con al menos 1.000 registros, Rusia (33,76 %), USA (29,90 %) y Alemania (23,65 %) tienen las tasas más altas, frente a 13,54 % del conjunto de otros países sin USA.
- **Edad:** los jóvenes adultos (18 a 34 años) tienen la mayor tasa (15,27 %), seguidos de los mayores de 35 (14,38 %). Los menores de 18 tienen la más baja (9,43 %).
- **Sexo:** las mujeres tienen una tasa ligeramente superior (15,10 % frente a 14,51 %). La diferencia es pequeña. El perfil de mayor tasa del cruce es mujer joven adulta (16,08 %).
- **Temporada:** en Verano la tasa es 15,32 % y en Invierno 11,73 %.
- **Número de pruebas:** la tasa baja al aumentar las pruebas por edición: 17,48 % con una prueba, 12,80 % con dos o tres y 9,40 % con cuatro o más.

**Limitación:** las medallas se cuentan por fila, así que en los deportes de equipo cada integrante suma una. Esto aumenta sobre todo las tasas de países con mucho deporte de equipo, como USA.

**Resumen:** los factores asociados a una mayor tasa de medalla son pertenecer a países como Rusia, USA o Alemania, ser joven adulto (18 a 34 años), competir en Juegos de Verano y participar en una sola prueba por edición. Las tasas más bajas se observan en los menores de 18 años, en los Juegos de Invierno y en quienes compiten en cuatro o más pruebas. Estos factores se analizaron por separado, sin medir su efecto combinado.